---
aliases:
  - Mathematical Foundations & Statistics for ML
  - Pure Python Statistics
  - ML Math Foundations
  - Statistical Thinking for ML
---

# Mathematical Foundations, Statistics & Probability for ML

# 0. How to Use This Note

This is a **thinking + retrieval system**, not a textbook.

For every concept, train through the full chain:

```text
CONCEPT
↓
INTUITION
↓
FORMULA
↓
MANUAL EXAMPLE
↓
PURE PYTHON
↓
NUMPY / PANDAS
↓
INTERPRETATION
↓
LIMITATIONS
↓
RETRIEVAL
```

`np.std(values)` is not enough for a reviewer. You need to be able to build it from scratch, explain every line, and say when the result would be misleading. This applies to **every** computational concept in this note, not just SD — mean, median, mode, variance, covariance, correlation, and the probability rules all get the same treatment.

**Five-Layer self-check** — before moving on from any concept, answer:

```text
WHAT?    → definition
WHY?     → what problem it solves
HOW?     → the maths
VERIFY   → manual + pure Python + library, all agreeing under the same
           definition/convention
WHEN NOT?→ when this result would mislead you
```

**Statistical Convention Rule** — before comparing results across manual, pure Python, and library, always ask: _population or sample? which denominator? which library default?_ Different numbers don't automatically mean a wrong calculation — check whether the conventions match first.

```python
np.var(values)             # population (÷N)
np.var(values, ddof=1)     # sample (÷n-1)
np.std(values)             # population
np.std(values, ddof=1)     # sample
df["score"].var()          # pandas default: sample
np.cov(x, y)[0, 1]                # sample covariance (np.cov returns a matrix)
np.cov(x, y, bias=True)[0, 1]     # population covariance
```

---

# PART 1 — Mathematical Foundations

## 1. Variables

```python
x = 10   # x is the variable, 10 is its current value
x = 20   # value can change
```

Datasets are collections of variables. Some become **features** (inputs), one becomes the **target** (what you predict).

**Numerical vs categorical** — a statistic must fit the type:

```text
mean(age)         → meaningful
mean(department)  → meaningless
```

**Units matter**: `175 cm`, `₹50,000`, `30 °C` — the unit is part of the number's meaning, and becomes critical when interpreting mean, variance, SD, slope.

## 2. Equations

```text
x + 5 = 12  →  x = 7
```

Think of it as a balance: whatever you do to one side, do to the other.

```python
x = 7    # assignment
x == 7   # comparison
```

## 3. Functions

```text
INPUT → FUNCTION → OUTPUT
f(x) = 2x + 3   →   f(5) = 13
```

```python
def f(x):
    return 2 * x + 3
```

**ML connection**: a model is a function. `y = wx + b`, where `x` = input, `w` = weight, `b` = bias/intercept, `y` = prediction.

## 4. Slope

```text
m = (y₂ - y₁) / (x₂ - x₁)
```

`(2,5)` → `(4,9)` gives `m = (9-5)/(4-2) = 2`: for every 1-unit increase in x, y increases by 2.

**Units of slope matter**: if x = study hours and y = score, slope has units "points per hour" — the number alone means nothing without them.

## 5. Intercept

For `y = mx + b`, `b` is the predicted value of y when `x = 0`.

**ML warning**: an intercept can be mathematically valid even when `x = 0` is unrealistic (e.g. age = 0). _Mathematically valid ≠ practically meaningful._

## 6. Graphs

Before interpreting any graph, check: what do x and y represent, what are the units, what does one point mean, **what is the axis scale?** A y-axis starting at 0 vs. starting at 95 can make the same data look completely different. Never interpret a graph before checking its scale.

## 7. Relationships Between Variables

```text
positive · negative · none obvious · nonlinear
```

For any relationship, ask: Is it linear? Curved? Are there clusters or outliers? Does it hold across the _whole_ dataset, or only part of it?

## 8. Mathematical Thinking for ML

```text
REAL PROBLEM → VARIABLES → RELATIONSHIPS → MATHEMATICAL REPRESENTATION
→ MODEL → PREDICTION → ERROR → IMPROVEMENT
```

Example: `price = f(size, bedrooms, location, age)` — the model learns `f`.

---

# PART 2 — Statistics: Center & Spread (Theory + Pure Python + Library)

## 9. Statistics Starts With a Question

|Question|Tool|
|---|---|
|What is typical?|Mean / Median|
|Most common?|Mode|
|How spread out?|SD / Variance / IQR|
|What does the shape look like?|Histogram|
|Possible outliers?|Boxplot / IQR|
|Two-variable relationship?|Scatter plot / Correlation|
|Uncertainty?|Probability|
|Sample → population?|Sampling analysis|

Don't reach for correlation just because you learned it recently — match the tool to the actual question.

## 10. Mean

```text
mean = Σx / n        e.g. [10,20,30] → 60/3 = 20
```

**Pure Python (explicit):**

```python
values = [10, 20, 30]
total = 0
for value in values:
    total += value
mean = total / len(values)
```

`sum()`/`len()` are just this loop pre-built. Compact form:

```python
mean = sum(values) / len(values)
```

**Library:** `np.mean(values)` · `df["score"].mean()`

**Sensitivity**: mean is pulled by extremes — `[10,20,30]` → mean 20; `[10,20,30,100]` → mean 40. One point moved it a lot.

**Sanity check**: mean must lie between min and max. A result outside that range means a bug.

## 11. Median

Middle value after sorting. `[3,7,9]` → 7. Even count: average the two middle values.

**Pure Python:**

```python
def median(values):
    values = sorted(values)
    n = len(values)
    if n % 2 == 1:
        return values[n // 2]
    return (values[n // 2 - 1] + values[n // 2]) / 2
```

`n // 2` is the middle _position_ (Python indexes from 0), which is why odd- length lists use it directly and even-length lists average the two flanking it.

**Why median?** Less sensitive to extremes: `[20,22,23,25,1000]` → median 23, but mean is dragged far above that.

## 12. Mode

Most frequent value. `[2,3,3,4,5]` → 3.

**Pure Python (no `Counter`), returns every tied top value:**

```python
def mode(values):
    counts = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    highest = max(counts.values())
    return [v for v, c in counts.items() if c == highest]
```

A dataset can be **unimodal** (one most-frequent value), **bimodal** (two tied for highest frequency), or **multimodal** (several tied). If every value occurs equally often, many definitions call that "no mode" — others say all values tie for highest frequency but there's no _unique_ one; this implementation returns the tied list either way. `mode([2,3,3,4,5,5])` → `[3, 5]` (bimodal).

## 13. Center & Spread — Comparison

|Statistic|Measures|Sensitive to outliers?|Units|
|---|---|---|---|
|Mean|arithmetic center|Yes|original|
|Median|middle position|Much less|original|
|Mode|most frequent|Depends|original|
|Variance|squared spread|Yes|squared|
|Std Dev|spread|Yes|original|

**Practical rule**: roughly symmetric → mean is fine; strongly skewed / extreme values present → median often describes center better. Guideline, not law.

## 14. Range

```python
range_value = max(values) - min(values)
```

Only uses two observations, so it's extremely sensitive to extremes.

## 15. Variance

```text
values → mean → deviation → square → average the squares
σ² = Σ(x-μ)²/N   (population)      s² = Σ(x-x̄)²/(n-1)   (sample)
```

Manual, `[8,10,12]`: mean=10, deviations `[-2,0,2]`, squared `[4,0,4]`, variance = `8/3`.

**Why square?** `-2+0+2 = 0` — unsquared deviations always cancel. Squaring keeps the magnitude.

**Pure Python (population):**

```python
def population_variance(values):
    m = sum(values) / len(values)
    return sum((v - m) ** 2 for v in values) / len(values)
```

**Sample** (divide by `n - 1` instead — used when the sample is estimating a population's variability, not describing itself):

```python
def sample_variance(values):
    m = sum(values) / len(values)
    return sum((v - m) ** 2 for v in values) / (len(values) - 1)
```

Library: `np.var(values)` (population) / `np.var(values, ddof=1)` (sample) · `df["score"].var()` (pandas defaults to sample).

**Two meanings of "variance"**: this section is _data_ variance — spread of observations. ML also uses _model_ variance — how much a model's predictions swing depending on training data (Part 7 §Bias vs Variance in ML). A reviewer asking "what is variance?" may expect you to name which one.

## 16. Standard Deviation

```text
σ = √variance
```

Why the square root? Variance is in squared units (cm²); the root returns it to the original scale (cm), which is interpretable.

**Pure Python:**

```python
def standard_deviation(values):
    m = sum(values) / len(values)
    variance = sum((v - m) ** 2 for v in values) / len(values)
    return variance ** 0.5   # x**0.5 == √x
```

(Sample version: swap the denominator to `len(values) - 1`.)

**Interpretation**: small SD → values cluster near the mean; large SD → more spread out. Always read it relative to scale/context — SD of 10 means very different things for "age" vs. "annual salary."

**Viva script** (say this while typing):

> "First the mean, because SD measures spread around it. Then each deviation. I square them so positive and negative don't cancel. I average the squares to get variance. Then square root to get back to the original units."

**Quick Q&A**

- Why subtract the mean? → measure each point's distance from center.
- Why n−1 for sample? → to correctly estimate population variability from a sample.
- What if there's 1 value? → population SD = 0; sample SD undefined (n−1 = 0).
- What if the list is empty? → undefined — nothing to summarize (mean/len both fail).
- Can SD or variance ever be negative? → No — both are built from squared terms.

## 17. Quartiles, Percentiles & IQR

```text
Q1 = 25th percentile   Q2 = median   Q3 = 75th percentile
IQR = Q3 - Q1   → spread of the middle 50%, less sensitive to extremes than range
```

```python
df["score"].quantile(0.25); df["score"].quantile(0.75)
```

**Outlier fences**: `lower = Q1 - 1.5×IQR`, `upper = Q3 + 1.5×IQR`. Flagged beyond these ≠ proven error.

```python
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = [v for v in values if v < lower or v > upper]
```

(Percentile calculation itself has more than one convention under the hood — know the concept cold; you're less likely to be asked to reimplement pandas' exact interpolation method.)

## 18. Outliers

An outlier is unusually distant — but that doesn't mean bad data. It could be measurement error, data-entry error, a rare-but-real event, or a valid extreme.

```text
FLAG → CHECK SOURCE → CHECK UNITS → CHECK DATA ENTRY
→ CHECK REAL-WORLD CONTEXT → DECIDE
```

Never auto-delete an outlier.

## 19. Variability Compared

```text
range     → full min-to-max span
IQR       → middle 50%
variance  → average squared deviation
std dev   → typical spread, in original units
```

No single measure is automatically "best" — they answer different questions.

## 20. Z-Scores & Standardization

```text
z = (x - μ) / σ
```

`mean=70, SD=10, x=90` → `z = (90-70)/10 = 2` → 90 is 2 SDs above the mean.

```python
z = (value - mean_value) / std
```

**Why it matters for ML**: features on very different scales (age 18–60 vs. salary 20,000–200,000) — for algorithms whose calculations depend on feature scale, leaving them unscaled can cause the larger-scale feature to have disproportionate influence. Standardizing (`z = (x-mean)/SD`) puts every feature on a comparable scale — but it does **not** make the data normally distributed, only rescales it.

---

# PART 3 — Pure Python Statistics Toolkit

Reviewer exams often require building these with **no numpy, pandas, statistics, or math module** — just loops, lists, dicts, `len()`, `sum()`.

**Why functions?** Packaging the logic once (`mean(values)`) makes it reusable across `mean(scores)`, `mean(ages)`, `mean(temperatures)` instead of retyping the formula every time.

**Edge cases to know:**

```python
def mean(values):
    if len(values) == 0:
        raise ValueError("values cannot be empty")   # division by zero otherwise
    return sum(values) / len(values)

def sample_variance(values):
    if len(values) < 2:
        raise ValueError("need at least 2 values")   # n - 1 must be ≥ 1
    m = mean(values)
    return sum((v - m) ** 2 for v in values) / (len(values) - 1)
```

**Complete toolkit:**

```python
def mean(values):
    return sum(values) / len(values)

def median(values):
    values = sorted(values)
    n = len(values)
    if n % 2 == 1:
        return values[n // 2]
    return (values[n // 2 - 1] + values[n // 2]) / 2

def mode(values):
    counts = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    highest = max(counts.values())
    return [v for v, c in counts.items() if c == highest]

def population_variance(values):
    m = mean(values)
    return sum((v - m) ** 2 for v in values) / len(values)

def sample_variance(values):
    m = mean(values)
    return sum((v - m) ** 2 for v in values) / (len(values) - 1)

def population_standard_deviation(values):
    return population_variance(values) ** 0.5

def sample_standard_deviation(values):
    return sample_variance(values) ** 0.5

# Test:
values = [8, 10, 10, 12, 15]
print("Mean:", mean(values))
print("Median:", median(values))
print("Mode:", mode(values))
print("Pop var / std:", population_variance(values), population_standard_deviation(values))
print("Sample var / std:", sample_variance(values), sample_standard_deviation(values))
```

**The three-level ladder to know for every statistic:**

```text
1. FORMULA     σ = √[Σ(x-μ)²/N]
2. PURE PYTHON mean → deviations → square → average → sqrt
3. LIBRARY     np.std(values)
```

The library isn't magic — it's the same algorithm, pre-built.

**Algorithm vs Implementation** — the reusable version of that idea, worth applying to every concept in this note:

```text
FORMULA      → what mathematical operation do we need?
ALGORITHM    → what sequence of steps produces that operation?
PURE PYTHON  → how do I implement those steps myself?
LIBRARY      → how does a library package the same idea for reuse/speed?
```

Worked for standard deviation:

```text
Formula     σ = √[Σ(x-μ)²/N]
Algorithm   mean → deviations → square → average → square root
Pure Python loops / arithmetic (the function above)
Library     np.std(values)
```

`np.std()` = algorithm (mean → deviation → square → average → root) + implementation (someone else's optimized code). You're being tested on the algorithm, not on whether you memorized the function name.

**Pure Python dependency map** — know what builds on what, so you can reconstruct a whole chain from one starting point:

```text
mean → variance → standard deviation
mean → covariance → correlation
quartiles (Q1, Q3) → IQR → outlier fences
probability → complement, independence, conditional → Bayes
```

**Translation drill**: when told _"find X without libraries,"_ say the steps out loud before coding — concept → formula → pure Python → (optionally) verify against `np.std`/`np.var`/`np.mean`.

---

# PART 4 — Probability (Theory + Pure Python)

## 21. Probability, Sample Space, Events

```text
0 ≤ P(A) ≤ 1        0=impossible, 1=certain
```

Die: `S = {1,2,3,4,5,6}`. Event `A` = even number = `{2,4,6}`. `P(A) = favorable / total` → `P(even) = 3/6`.

## 22. Complement

```text
P(not A) = 1 - P(A)
```

```python
p_red = 5 / 10
p_not_red = 1 - p_red
```

## 23. Addition Rule

```text
P(A or B) = P(A) + P(B) - P(A and B)
```

Subtracting the overlap avoids double-counting shared outcomes.

## 24. Independent Events

```text
P(A and B) = P(A) × P(B)      (if independent)
```

```python
p_six = 1 / 6
p_both_six = p_six * p_six
```

**Misconception check**: independent ≠ mutually exclusive. Two independent coin flips can both land heads — they still _happen together_.

## 25. Conditional Probability

```text
P(A|B) = P(A and B) / P(B)
```

The condition **changes the reference group**. If 100 students total but you're asking "P(passed | department A)," you're now only considering the 40 in department A — not all 100.

```python
python_students = 60
python_and_passed = 45
p_passed_given_python = python_and_passed / python_students
```

## 26. Bayes' Theorem

```text
P(A|B) = P(B|A) P(A) / P(B)
```

Vocabulary: `P(A)` = **prior**, `P(B|A)` = **likelihood**, `P(B)` = **evidence**, `P(A|B)` = **posterior**. Mental model: `prior → new evidence → updated belief`.

```python
p_a_given_b = (p_b_given_a * p_a) / p_b
```

**From counts** — often more intuitive than the formula: 1,000 people, 100 have the condition, 900 don't; 80 of the 100 affected test positive, 45 of the 900 unaffected also test positive.

```python
true_positive = 80
false_positive = 45
p_condition_given_positive = true_positive / (true_positive + false_positive)
```

Same idea as the base-rate intuition below, made concrete with numbers.

## 27. Base-Rate Intuition

If a disease is rare, even a fairly accurate test can produce more false positives than true positives in absolute terms, because the pool of healthy people is so much larger. This is exactly why `P(test+|disease)` ≠ `P(disease|test+)` — one of the most important probability lessons for ML.

## 28. Expected Value

```text
E[X] = Σ x·P(x)
```

Game paying ₹100 (p=0.2) or ₹0 (p=0.8): `E[X] = 100(0.2)+0(0.8) = 20`. Over many repetitions the average approaches ₹20 — a single trial will not.

```python
outcomes = [100, 0]
probs = [0.2, 0.8]
expected_value = sum(x * p for x, p in zip(outcomes, probs))
```

---

# PART 5 — Distributions

## 29. Distribution Basics

For any distribution ask: where's the center, how wide is the spread, is it symmetric, skewed, are there outliers, multiple peaks, gaps?

## 30. Histogram

Groups values into bins and counts observations per bin. **Bin count changes the visual shape** — `bins=5` vs `bins=30` can look different for the same data. Don't over-interpret small visual details caused by bin choice.

## 31. Normal Distribution

Symmetric, bell-shaped, single-peaked; `mean ≈ median ≈ mode`.

```text
68% within ±1 SD · 95% within ±2 SD · 99.7% within ±3 SD
```

**Trap**: "numerical data" does not imply "normally distributed." Real data can be skewed, multi-modal, heavy-tailed, or bounded — always inspect.

## 32. Skewness

```text
Right-skewed: long tail right, often mean > median
Left-skewed:  long tail left,  often mean < median
```

Extreme values pull the mean toward the tail; median resists this better.

---

# PART 6 — Correlation & Relationships

## 33. Positive / Negative Correlation

x↑ with y↑ = positive; x↑ with y↓ = negative.

## 34. Pearson Correlation

```text
-1 ≤ r ≤ 1     +1 perfect positive · 0 no linear association · -1 perfect negative
```

`r=0.92` → strong positive; `r=-0.81` → strong negative; `r=0.03` → little linear association. **No universal cutoff** ("0.7 = strong") — sample size, measurement quality, and domain context all matter.

**Sanity check**: `r` must satisfy `-1 ≤ r ≤ 1` — a result like `1.3` means a bug, not a stronger relationship.

## 35. Covariance

Covariance asks whether two numerical variables tend to move together — same idea as variance, but across two variables instead of one.

```text
Cov(X,Y) = Σ[(x-x̄)(y-ȳ)] / N
```

```python
def covariance(x, y):
    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    if len(x) == 0:
        raise ValueError("values cannot be empty")

    mean_x = sum(x) / len(x)
    mean_y = sum(y) / len(y)
    return sum((xv - mean_x) * (yv - mean_y) for xv, yv in zip(x, y)) / len(x)
```

Positive → they tend to rise together; negative → one rises as the other falls; near zero → little linear co-movement. Unlike correlation, covariance has no fixed scale — its size depends on the units of x and y (e.g. x in cm, y in kg gives covariance in cm·kg), which is exactly why correlation standardizes it.

**Population vs sample, and `np.cov`**: the function above divides by `N` — population covariance. `np.cov(x, y)` defaults to **sample** covariance (÷ n-1); pass `bias=True` to get the population version to match this function. `np.cov` also returns a **2×2 matrix**, not a single number, so index into it to compare against the scalar this function returns:

```text
[0,0] → variance of x     [0,1] → covariance of x,y
[1,0] → covariance of y,x [1,1] → variance of y
```

```python
np.cov(x, y)[0, 1]                # sample covariance — pandas/most libraries' default
np.cov(x, y, bias=True)[0, 1]     # population covariance — matches the function above
```

Same Statistical Convention Rule as everywhere else in this note: check the denominator (and remember the matrix shape) before assuming a mismatch is an error.

## 36. Pearson Correlation From Scratch

```text
r = Σ[(x-x̄)(y-ȳ)] / √[Σ(x-x̄)² · Σ(y-ȳ)²]
```

Equivalently: `r = covariance(x,y) / (SD(x) · SD(y))` — correlation is covariance rescaled into a fixed −1..1 range.

```python
def correlation(x, y):
    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    if len(x) == 0:
        raise ValueError("values cannot be empty")

    mean_x = sum(x) / len(x)
    mean_y = sum(y) / len(y)

    numerator = 0
    sum_sq_x = 0
    sum_sq_y = 0

    for xv, yv in zip(x, y):
        dx = xv - mean_x
        dy = yv - mean_y
        numerator += dx * dy
        sum_sq_x += dx ** 2
        sum_sq_y += dy ** 2

    denominator = (sum_sq_x * sum_sq_y) ** 0.5
    if denominator == 0:
        raise ValueError("correlation is undefined when either variable has zero variance")

    return numerator / denominator
```

Reviewer mental model: x-deviations and y-deviations → multiply pairwise (joint movement) → normalize by each variable's own spread. Verify against `df["x"].corr(df["y"])`.

**Covariance vs correlation** — the distinction a reviewer will probe:

```text
COVARIANCE                          CORRELATION
direction of joint movement         standardized covariance
has units (x's unit × y's unit)     unitless
depends on the units of x and y     always between -1 and +1
hard to compare across datasets     easy to compare across datasets
```

Height/weight in cm+kg vs. in inches+lbs will give _different_ covariance values purely from the unit change. Under ordinary positive unit conversions (like cm→inches), Pearson correlation keeps the same value, because both variables are rescaled consistently and that scaling cancels out in the standardization.

## 37. Correlation Is Not Causation

Ice cream sales ↑ and sunburn ↑ correlate — both are driven by hot weather, not by each other. A correlation can come from: causation, a confounder, reverse causality, or coincidence.

## 38. Confounding Variables

A confounder relates to both variables and can create or distort their apparent relationship (temperature → ice cream sales _and_ → sunburn). Ignoring it can make an X–Y relationship look misleading.

## 39. Nonlinear Relationships

Pearson mainly measures _linear_ association — a strong curved relationship can still produce `r ≈ 0`. Read `r≈0` as "little linear association," not "no relationship at all."

## 40. Feature Relationships in ML

Features can overlap in the information they carry (age ↔ experience ↔ salary). This affects interpretability, model coefficients, and multicollinearity — worth checking before assuming each feature is independent evidence.

---

# PART 7 — Sampling & Statistical Thinking for ML

## 41. Population vs Sample

Population = the complete group of interest; sample = the subset you actually observed. We study samples because measuring the whole population is usually expensive, slow, or impossible.

## 42. Parameter vs Statistic

```text
population → parameter (μ)      sample → statistic (x̄)
```

## 43. Sampling Variability

Different random samples from the same population give different sample means — the population didn't change, the sample did.

## 44. Sampling Bias

Bias occurs when the collection process itself systematically misrepresents the population (e.g. only surveying students already inside the cafeteria about cafeteria food). A **huge but biased** sample is still biased — size doesn't fix a broken sampling design.

## 45. Representative Sample

A sample can follow an unbiased method and still miss groups. Ask:

```text
population → sampling method → who got included? → who got excluded?
→ any systematic difference between them? → representativeness
```

This is "sampling bias" applied to real datasets: a training set scraped from one source, one region, or one time window may not represent the population your model will actually see in production.

## 46. Random Error vs Systematic Bias

```text
random error     → unpredictable variation around the true value
systematic bias  → a consistent offset (e.g. a scale that always reads +5kg)
```

## 47. Bias vs Variance in ML

Different use of the words from dataset variance in Part 2:

```text
model bias     high → underfitting (model too simple to capture the pattern)
model variance high → overfitting (model too sensitive to the training data)
```

This reuses "variance" in a different sense than Part 2 — there it meant spread of observations (_data_ variance); here it means how much a model's predictions swing depending on training data (_model_ variance). A reviewer asking "what is variance?" may expect you to identify which one.

## 48. Generalization

The goal isn't memorizing training data — it's performing well on **unseen** data. Training performance ≠ real-world performance.

---

# PART 8 — Sanity Checks & Choosing the Right Tool

## 49. Sanity Checks

```text
mean must be between min and max
0 ≤ probability ≤ 1
-1 ≤ correlation r ≤ 1
0.25 = 25%   (not 0.25%)
SD reported in cm², not cm → someone reported variance instead
```

Cheap checks that catch real mistakes immediately.

## 50. The Statistic Selection Tree

```text
What am I asking?
├── typical value?      → mean / median
├── most common?         → mode
├── spread?              → range / variance / SD / IQR
├── shape?                → histogram / boxplot
├── relation?             → scatter plot / correlation
├── uncertainty?          → probability / conditional / Bayes
└── sample → population?  → sampling / bias / uncertainty
```

## 51. "Why This, Not That?" Drill

Before calculating, finish the sentence: _"I'm using ___ because I want to measure ___."_ And be ready to defend the choice: _"Why median instead of mean?"_ → data is skewed, extreme values present, median is less sensitive. _"Why correlation instead of just the mean?"_ → mean describes one variable, correlation describes the linear association between two.

---

# PART 9 — Hands-On Dataset & Workflow

## 52. Practice Dataset

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "student": ["Ali","Sara","John","Maya","David","Asha","Ravi","Noah","Lina","Zain"],
    "department": ["A","B","A","B","A","C","C","B","A","C"],
    "study_hours": [2,4,5,6,3,7,1,8,4,6],
    "attendance": [70,82,88,91,75,95,60,98,84,90],
    "score": [55,68,74,82,61,91,45,96,72,85],
})
```

## 53. New-Dataset Protocol

```text
READ THE QUESTION → UNDERSTAND VARIABLES → CHECK UNITS → INSPECT
→ CHECK MISSING/TYPES → CHECK DISTRIBUTIONS/OUTLIERS → CHOOSE STATISTICS
→ CHECK RELATIONSHIPS → VISUALIZE → INTERPRET → QUESTION ASSUMPTIONS → CONCLUDE
```

Before calculating anything, ask: what does one row represent? How was it collected? Could there be selection bias? What's the sample size?

## 54. Inspect → Summarize → Relate → Group

```python
df.head(); df.shape; df.columns; df.dtypes; df.info(); df.describe(); df.isna().sum()

df["score"].mean(); df["score"].median(); df["score"].std()

df["study_hours"].corr(df["score"])
df["attendance"].corr(df["score"])

df.groupby("department")["score"].agg(["mean", "median", "std", "min", "max"])
```

Don't stop at "Department A scored higher" → that's descriptive, not causal, unless you have stronger evidence.

## 55. "Never Stop at the Number"

Bad: `mean = 74`. Better: _"The average score is 74, with a standard deviation of 9, so scores vary moderately around that center — but this alone doesn't describe the full shape of the distribution."_

```text
STATISTIC + DATASET + DIRECTION/MAGNITUDE + UNITS + LIMITATION
```

---

# PART 10 — Visualization

```python
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(df["score"], bins=5, edgecolor="black")
ax.set_title("Score Distribution"); ax.set_xlabel("Score"); ax.set_ylabel("Frequency")
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.boxplot(df["score"]); ax.set_title("Score Distribution"); ax.set_ylabel("Score")
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df["study_hours"], df["score"])
ax.set_title("Study Hours vs Score"); ax.set_xlabel("Study Hours"); ax.set_ylabel("Score")
plt.show()
```

---

# PART 11 — AI-Assisted Learning

**For explanation** — give AI constraints, not just "explain X":

```text
Explain standard deviation assuming I know mean.
Use [8,10,12]. Show every calculation. Explain why we square, then why
we square-root. Connect it to ML. Give 3 misconceptions.
```

**For retrieval** — make it quiz you, not lecture you:

```text
Give me one problem at a time. Don't reveal the answer. Make me name the
concept, then the formula, then calculate, then interpret. Check my reasoning.
```

**As an adversary + verification loop** — after reaching a conclusion:

```text
Try to disprove my interpretation. List alternative explanations and
hidden assumptions. Tell me what analysis would test them.
```

```text
AI explanation → manual reasoning → Python calculation → compare
→ inspect graph → question assumptions → final interpretation
```

The goal is exposing weak reasoning, not outsourcing the conclusion.

---

# PART 12 — Common Misconceptions

```text
Mean = "typical value"?           No — it's the arithmetic average.
Median always better than mean?   No — depends on skew/outliers.
SD = "error"?                     No — it's spread, not measurement error.
Outlier = bad data?               No — investigate before deciding.
Correlation = causation?          No — needs stronger evidence/design.
r ≈ 0 = no relationship?          No — just no *linear* one detected.
80% probability = it will happen? No — describes uncertainty, not certainty.
Bigger sample = automatically unbiased? No — a huge biased sample stays biased.
Independent = mutually exclusive? No — independent events can co-occur.
```

---

# PART 13 — Reviewer Prep: Drills, Retrieval Levels & Trick Questions

## 56. Pure-Python Drills (no numpy/pandas/statistics/math)

```text
1. Mean of [12,18,20,25,30] — with sum()/len(), then again with a manual loop.
2. Median of [7,2,10,4,8], then [7,2,10,4,8,20] (odd vs even count).
3. Mode of [2,4,4,5,5,5,7,8] without collections.Counter — return all tied modes.
4. Population variance of [5,7,9,11,13].
5. Population AND sample standard deviation of [5,7,9,11,13] — explain n vs n-1.
6. Write one `standard_deviation(values)` function using only loops/lists/len/sum.
7. Covariance and correlation of x=[1,2,3,4,5], y=[2,4,5,8,10] — verify with df.corr().
8. Z-score of x=90 given mean=70, std=10.
9. Given q1 and q3, compute IQR and the outlier fences, then filter a list against them.
10. Reviewer says "you forgot the SD code — rebuild it from the formula." Do it cold.
```

## 57. Retrieval Levels (apply to any topic above)

```text
1. RECOGNITION   → read df["score"].mean(), explain what it does
2. PRODUCTION    → "find the average score" → write the syntax unaided
3. SELECTION     → "data is skewed — which measure of center, and why?"
4. INTERPRETATION→ mean=75, median=68 — what might that suggest?
5. CRITIQUE      → "corr(attendance,score)=0.82, so attendance causes high
                    scores" — evaluate that claim
```

Level 5 is the actual target — recognition alone doesn't count.

## 58. Cold Retrieval (no notes, by topic)

```text
Math:        variable · equation · function · slope · intercept · y=mx+b · y=wx+b in ML
Stats:       mean · median · mode · variance · why square · why sqrt · IQR · z-score
Pure Python: rebuild mean/median/mode/pop-var/sample-var/pop-std/sample-std/
             covariance/correlation from scratch
Probability: event · complement · independence · P(A|B) · Bayes · expected value
Distributions: normal · skew · outlier · SD · IQR · histogram vs boxplot
Correlation: positive/negative · Pearson · covariance vs correlation · confounder ·
             why r≈0 can still hide a relationship
Sampling:    population · sample · parameter · statistic · representativeness ·
             sampling bias vs random error
```

## 59. Reviewer Mode & Attack Zone

```text
Don't jump into code. First name the concept → then the formula
→ then write code → then explain the output.
```

Example — _"Find SD without libraries":_

> "This is standard deviation. I need the mean first, then each deviation from it, squared, averaged for variance, then square-rooted."

```python
mean = sum(values) / len(values)
variance = sum((x - mean) ** 2 for x in values) / len(values)
std = variance ** 0.5
```

> "This is _population_ SD because I divided by n, not n-1."

**Then try to disprove your own result** — the step most people skip:

> "Correlation between attendance and score is 0.82. Could something other than a direct effect explain that? Maybe both are driven by motivation, or a third factor like study hours." That question — not the number — is the actual statistical thinking a reviewer is testing for.

**Reviewer Attack Zone** — questions that test understanding, not recall:

- _Why square the deviation instead of using absolute value?_ → squaring is mathematically convenient (differentiable); mean absolute deviation is a separate, less common statistic that also avoids cancellation.
- _Why n−1 for sample variance?_ → corrects for estimating population variability from a sample rather than describing the sample itself.
- _Why square root after variance?_ → returns squared units to the original scale.
- _Why is median resistant to outliers?_ → it depends only on rank/position, not magnitude.
- _Can SD or variance ever be negative?_ → No — both come from squared terms.
- _Can correlation be greater than 1?_ → No — `-1 ≤ r ≤ 1` by construction; outside that means a bug.
- _Can r ≈ 0 while variables are clearly related?_ → Yes — Pearson only detects _linear_ association.
- _Does standardization make data normally distributed?_ → No — it only rescales; shape is unchanged.
- _Why doesn't a bigger sample automatically remove bias?_ → size doesn't fix a flawed sampling method.
- _Can two datasets share a mean but have very different SDs?_ → Yes — `[10,10,10]` and `[0,10,20]` both mean 10.
- _Can two variables be highly correlated without one causing the other?_ → Yes — confounder, reverse causality, or coincidence.

## 60. Reviewer Translation Protocol — Exam Emergency Map

The instant translation from spoken request to first step, for when the screen is staring at you and your mind goes blank:

```text
"Find the mean without libraries"        → total / count
"Find median without libraries"          → sort → middle
"Find mode without libraries"            → frequency dict → highest frequency
"Find variance without libraries"        → mean → deviations → squares → average
"Find standard deviation without libraries" → variance → square root
"Find covariance without libraries"      → two means → paired deviations → multiply → average
"Find correlation without libraries"     → covariance → normalize by both standard deviations
"Find z-score without libraries"         → (value - mean) / standard deviation
"Find probability"                       → favorable / total
"Find conditional probability"           → intersection / condition
"Apply Bayes"                            → posterior = likelihood × prior / evidence
```

---

# PART 14 — Leech Log

|Date|Concept|What I forgot|Correct answer|Repetitions|
|---|---|---|---|---|
||||||

**Rule**: forgot once → review. Forgot twice → it's a leech, drill it 5 min with varied numbers/columns. Forgot three times → deep rebuild: explain from memory → derive formula → solve manually → pure Python → verify with library → explain a misconception and an edge case.

---

# PART 15 — Formula Survival Sheet

```text
y = mx + b                    m = (y₂-y₁)/(x₂-x₁)
mean = Σx/n
σ² = Σ(x-μ)²/N   (population)      s² = Σ(x-x̄)²/(n-1)   (sample)
σ = √variance                       z = (x-μ)/σ
range = max - min                   IQR = Q3 - Q1
outlier fences = Q1 - 1.5·IQR , Q3 + 1.5·IQR
Cov(X,Y) = Σ[(x-x̄)(y-ȳ)]/N
r = Cov(X,Y) / (SD(x)·SD(y))    (Pearson correlation)
P(not A) = 1 - P(A)
P(A or B) = P(A) + P(B) - P(A and B)
P(A and B) = P(A)P(B)                (independent)
P(A|B) = P(A and B)/P(B)
P(A|B) = P(B|A)P(A)/P(B)             (Bayes)
E[X] = Σ x·P(x)
```

---

# PART 16 — Python Survival Sheet

```python
# Pure Python
sum(values); len(values); sorted(values); min(values); max(values)
mean = sum(values) / len(values)
variance = sum((x - mean) ** 2 for x in values) / len(values)
std = variance ** 0.5
mean_x = sum(x) / len(x); mean_y = sum(y) / len(y)
std_x = (sum((v - mean_x) ** 2 for v in x) / len(x)) ** 0.5
std_y = (sum((v - mean_y) ** 2 for v in y) / len(y)) ** 0.5
covariance = sum((xv - mean_x) * (yv - mean_y) for xv, yv in zip(x, y)) / len(x)
correlation = covariance / (std_x * std_y)

# NumPy
np.mean(values); np.median(values)
np.var(values); np.var(values, ddof=1)      # population / sample
np.std(values); np.std(values, ddof=1)
covariance = np.cov(x, y, bias=True)[0, 1]   # population covariance, as a single number
np.cov(x, y)[0, 1]                           # sample covariance, same [0,1] indexing

# Pandas
df["score"].mean(); df["score"].median(); df["score"].mode()
df["score"].var(); df["score"].std()
df["score"].quantile(0.25); df["score"].quantile(0.75)
df["height"].corr(df["weight"])
df.groupby("group")["value"].agg(["mean", "min", "max"])
```

---

# PART 17 — Final Mental Model

```text
MATH → variables, equations, functions, slope, intercept, graphs, relationships
STATS → center (mean/median/mode), spread (range/variance/SD/IQR),
        distributions, covariance & correlation
PURE PYTHON → same stats, built from loops/lists/dicts/functions
PROBABILITY → events, complement, independence, conditional, Bayes, expected value
STATISTICAL THINKING → population/sample, parameter/statistic, representativeness,
                        bias, variance, generalization
DATA ANALYSIS → inspect, summarize, visualize, compare, interpret, verify
    ↓
MACHINE LEARNING → features, targets, prediction, generalization, uncertainty
```

# The Actual Objective

Don't memorize the notebook. Build this reflex:

```text
QUESTION
↓ IDENTIFY DATA TYPE
↓ IDENTIFY STATISTICAL QUESTION
↓ CHOOSE TOOL (and be able to explain why this one, not another)
↓ FORMULA
↓ MANUAL CALCULATION
↓ PURE PYTHON
↓ LIBRARY VERIFICATION
↓ SANITY / UNIT CHECK
↓ VISUAL CHECK
↓ INTERPRET
↓ QUESTION ASSUMPTIONS
↓ TRY TO DISPROVE THE RESULT
↓ CONCLUDE
```

`np.std(values)` proves you know the function name. Building it from scratch, explaining every line, and then trying to poke holes in your own conclusion — that's the difference between knowing a function and thinking statistically, and it's what a reviewer is actually testing for.